<a href="https://colab.research.google.com/github/rampellisaieshwar/Convolutional_Neural_Network-CNN-Architectures/blob/main/DenseNet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Introduction to DenseNet
DenseNet improves information flow between layers by connecting every layer directly to every subsequent layer. Key benefits include:
* **Alleviated Gradient Vanishing**: Short paths from early layers to the loss function.
* **Feature Reuse**: Layers can access features from any previous layer.
* **Parameter Efficiency**: Requires fewer parameters than ResNet because it doesn't need to relearn redundant feature maps.

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torchvision.models import densenet121

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Data augmentation and normalization
transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

# Load CIFAR-10
trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=64, shuffle=True)

testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=64, shuffle=False)

100%|██████████| 170M/170M [00:10<00:00, 16.1MB/s]


In [2]:
# Initialize DenseNet121 and modify the final layer for 10 classes
model = densenet121(num_classes=10).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print(f"Training on: {device}")

Training on: cpu


In [3]:
# Training loop (Simplified for demonstration: 2 epochs)
num_epochs = 2
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for i, (images, labels) in enumerate(trainloader):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss/len(trainloader):.4f}')

Epoch [1/2], Loss: 1.5349
Epoch [2/2], Loss: 1.1523


In [4]:
# Evaluation
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for images, labels in testloader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f'Accuracy of the network on the 10000 test images: {accuracy:.2f}%')

Accuracy of the network on the 10000 test images: 60.24%


### Conclusion
Based on the results:
1. **Performance**: Even with minimal training, DenseNet shows high accuracy due to its efficient feature propagation.
2. **Complexity**: While conceptually complex due to dense connections, it is memory efficient in terms of the number of parameters.
3. **Application**: DenseNet is ideal for tasks where training data might be limited or where high-frequency feature reuse is beneficial.